# Freight Rate Prediction — Analysis

Working notebook for the Spotter ML assessment. The task is to predict `posted_rate` (USD) for spot freight loads using LightGBM trained on historical January–October 2025 data.

## 1. Problem setup

- **Target:** `posted_rate` — the spot rate in USD for a full truckload shipment
- **Development period:** 48,000 loads from January–October 2025 (labelled)
- **Validation period:** 12,000 loads from November–December 2025 (unlabelled)
- **December benchmark:** 31-day fixed-corridor prediction (Lexington → Fort Wayne)

## 2. Data overview

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../train-test.csv")
df["date"] = pd.to_datetime(df["date"])

print(df.shape)
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df.date.min().date()} to {df.date.max().date()}")
print(f"\nposted_rate summary:")
print(df["posted_rate"].describe())

(48000, 14)
Columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']

Date range: 2025-01-01 to 2025-10-31

posted_rate summary:
count    48000.000000
mean      2373.980682
std       1486.493245
min         57.220000
25%       1251.555000
50%       2030.760000
75%       3330.750000
max      25533.000000
Name: posted_rate, dtype: float64


In [2]:
print(df["equipment"].value_counts())

equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64


48K labelled loads, 14 columns. Dry Van is the most common equipment type (~57%).

The dataset includes `market_index` and `quote_signal` columns, but these were not used in the final model — they are not available for the December benchmark inputs.

## 3. Quick EDA

### Posted rate distribution

![Target Distribution](../assets/target_distribution.png)

Right-skewed with a long tail. Median ($2,031) is well below the mean ($2,374), pulled up by high-value loads.

### Distance vs posted rate

![Distance vs Rate](../assets/distance_vs_rate.png)

Distance has a strong positive relationship with posted rate. The spread also increases for longer routes.

### Rate by equipment type

In [3]:
equip_stats = df.groupby("equipment")["posted_rate"].agg(["mean", "median", "count"])
equip_stats["mean"] = equip_stats["mean"].round(0).astype(int)
equip_stats["median"] = equip_stats["median"].round(0).astype(int)
print(equip_stats)

            mean  median  count
equipment                      
Dry Van     2272    1953  27202
Flatbed     2445    2077   8753
Reefer      2554    2197  12045


Reefer loads pay the highest average rate, followed by Flatbed. This is expected — refrigerated and flatbed equipment carry cost premiums.

## 4. Data quality

In [4]:
print(f"Missing weights:  {df['weight'].isna().sum()}")
print(f"Negative weights: {(df['weight'] < 0).sum()}")
print(f"Zero weights:     {(df['weight'] == 0).sum()}")
print(f"Missing coords:   {df['pickup_lat'].isna().sum()}")

Missing weights:  300
Negative weights: 292
Zero weights:     0
Missing coords:   0


300 missing weights and 292 negative weights (~1.2% of loads total).

I checked the absolute-value distribution of negative weights against positive weights for the same equipment types — they match closely, which supports treating negatives as sign errors rather than a separate category.

**Resolution in `src/preprocessing.py`:**
- Take absolute value of all weights
- Impute missing weights with equipment-group median from training data
- All imputation parameters are fitted on training data only

In [5]:
val = pd.read_csv("../validation.csv")

train_pickups = set(df["pickup"].unique())
train_deliveries = set(df["delivery"].unique())
val_pickups = set(val["pickup"].unique())
val_deliveries = set(val["delivery"].unique())

print(f"Train unique pickups:    {len(train_pickups)}")
print(f"Train unique deliveries: {len(train_deliveries)}")
print(f"Val unseen pickups:      {len(val_pickups - train_pickups)}")
print(f"Val unseen deliveries:   {len(val_deliveries - train_deliveries)}")

Train unique pickups:    64
Train unique deliveries: 64
Val unseen pickups:      8
Val unseen deliveries:   8


8 pickup and 8 delivery cities in the validation set are unseen in training. Target encoding falls back to the global training mean for unseen cities, while geographic coordinates from the validation inputs are still available for spatial features.

## 5. Feature engineering

The final feature set has 33 engineered features across these groups:

| Group | Features | Notes |
|-------|----------|-------|
| Distance | `distance`, `log_distance`, `distance_sq` | Raw + transforms |
| Weight | `weight`, `log_weight` | After abs + imputation |
| Equipment | `equipment_code`, `equip_premium` | Ordinal + rate multiplier |
| Calendar | `day_of_week`, `day_of_month`, `is_weekend`, sin/cos encodings | Cyclical encoding for periodicity |
| Spatial | lat/lon pairs, `haversine_dist`, `circuity`, deltas, midpoints | 12 features |
| Interactions | `dist_x_weight`, `dist_x_premium`, `log_dist_x_premium` | Cross-features |
| Target enc. | `pickup_target_enc`, `delivery_target_enc`, `equip_rpm_enc`, `est_baseline_rate` | OOF encoded |

Implementation is in `src/features.py` and `src/encoding.py`.

In [6]:
from src.preprocessing import FeaturePipeline

pipeline = FeaturePipeline()
pipeline.fit(df)
oof = pipeline.compute_oof_target_encodings(df, n_splits=5)
X = pipeline.transform(df, is_training=True, oof_encodings=oof)

print(f"Feature count: {len(X.columns)}")
print(list(X.columns))

Feature count: 33
['distance', 'log_distance', 'distance_sq', 'weight', 'log_weight', 'equipment_code', 'equip_premium', 'day_of_week', 'day_of_month', 'is_weekend', 'dow_sin', 'dow_cos', 'dom_sin', 'dom_cos', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'haversine_dist', 'circuity', 'delta_lat', 'delta_lon', 'abs_delta_lat', 'abs_delta_lon', 'mid_lat', 'mid_lon', 'dist_x_weight', 'dist_x_premium', 'log_dist_x_premium', 'pickup_target_enc', 'delivery_target_enc', 'equip_rpm_enc', 'est_baseline_rate']


## 6. Model experiments

I used LightGBM throughout — it trains fast on 48K rows and handles the mix of continuous/categorical features well.

Key decisions during development:

- **`market_index` and `quote_signal` were removed** from the feature set. These columns are present in `train-test.csv` but not in `december-chart-inputs.csv`, so including them would require fabricating December values. Removing them also slightly reduced CV error in my experiments.
- **Target encoding over one-hot:** The city columns have 64+ unique values. OOF target encoding captures rate-level information without blowing up dimensionality.
- **Cyclical calendar encoding** (sin/cos) instead of raw day-of-week ordinals — preserves the fact that day 6 and day 0 are adjacent.
- **`est_baseline_rate = distance * equip_rpm_enc`** gives the tree a strong linear prior to split around.

Final LightGBM config (from `src/config.py`):

In [7]:
from src.config import MODEL_PARAMS

for k in ["num_leaves", "learning_rate", "feature_fraction", "bagging_fraction",
          "min_child_samples", "reg_alpha", "reg_lambda"]:
    print(f"{k+':':22s}{MODEL_PARAMS[k]}")

num_leaves:       63
learning_rate:    0.03
feature_fraction: 0.85
bagging_fraction: 0.85
min_child_samples: 20
reg_alpha:        0.1
reg_lambda:       0.1


Tree count is determined by early stopping during CV (patience=150), then the final model uses 1.25x the average best iteration.

## 7. Validation

Random splitting would leak future data into training. I used expanding-window time-series CV instead:

| Fold | Train | Validate |
|------|-------|----------|
| 1 | Jan-Jul | Aug |
| 2 | Jan-Aug | Sep |
| 3 | Jan-Sep | Oct |
| **Holdout** | **Jan-Aug** | **Sep-Oct** |

The 2-month holdout (Sep-Oct) mirrors the actual Nov-Dec forecast horizon.

Preprocessing is re-fitted inside each fold — no information from the validation partition leaks into imputation or target encoding statistics.

### Target encoding

Route-level target statistics (`pickup_target_enc`, `delivery_target_enc`) are calculated out-of-fold during training so a row never uses its own target in the encoding. I used 5-fold OOF with Bayesian smoothing (m=15).

At inference, encoding statistics are computed from the full training set, with global mean fallback for unseen cities.

### Results

![Validation Performance](../assets/validation_performance.png)

## 8. Final results

| | MAE | RMSE | R² | MAPE | MedAE |
|---|---|---|---|---|---|
| **CV 3-fold avg** | $130.02 | $632.87 | 0.823939 | 6.14% | $47.23 |
| **Sep-Oct holdout** | $129.48 | $635.27 | 0.826706 | 5.96% | $44.84 |

Per-fold MAE: Fold 1 = $133.78, Fold 2 = $126.22, Fold 3 = $130.07.

The holdout MAE ($129.48) is close to the CV average ($130.02), which is a good sign for stability. The gap between RMSE and MAE reflects the heavy-tailed target distribution.

These are internal development results. The official validation score is calculated by Spotter after submission.

## 9. December benchmark

Fixed corridor: Lexington > Fort Wayne, 360 miles, Dry Van, 32,000 lbs, 31 days.

In [8]:
dec = pd.read_csv("../december_predictions.csv")
print(f"Min:    ${dec['predicted_rate'].min():.2f}")
print(f"Max:    ${dec['predicted_rate'].max():.2f}")
print(f"Mean:   ${dec['predicted_rate'].mean():.2f}")
print(f"Spread: ${dec['predicted_rate'].max() - dec['predicted_rate'].min():.2f}")

Min:    $842.77
Max:    $848.53
Mean:   $846.05
Spread: $5.75


![December Forecast](../assets/december_forecast.png)

The December output shows a small weekday/weekend pattern from the calendar features. The overall range is narrow (~$6) since everything except the date is held constant on this corridor.

Holiday-specific effects (Christmas, New Year's) are not directly represented in the training period, which ends in October.

## 10. Takeaways

- Distance is the strongest pricing signal, supplemented by spatial geometry and equipment type.
- Time-based validation is more appropriate than random splitting since the final predictions cover a future period.
- OOF target encoding avoids in-sample target leakage on high-cardinality city columns.
- All preprocessing parameters are fitted on training data only and frozen before inference.
- The supplied `score.py` scorer passed with exit code 0 — 12,000 validation predictions + 31 December predictions, all valid.